# Elastodynamic MHM: spatial and temporal convergence

The local Newmark responses and global traction coupling implement Gomes et al. (2017), [Equations (42)–(48)](https://doi.org/10.20906/CPS/CILAMCE2017-0399). The cube study uses the analytical displacement in Equation (53), with $\lambda=\mu=0.4$, $\rho=1$, and zero initial displacement and velocity. Its source is derived independently from $\rho u_{tt}-\nabla\cdot\sigma(u)$.

The numerical settings are local P3 fields, linear vector face traces, two local subdivisions, $6n^3$ macro tetrahedra and $T=0.5$. The seven spatial levels use $n=1,2,3,4,5,6,8$ and $\Delta t=0.005$. The paper does not unambiguously specify the local degree or historical connectivity, so these declared choices do not identify identical historical matrices.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.meshes.triangle import TriangleMesh

data = root / "examples/results/elastodynamics"

## User-defined local forms and actual subcycling

The primary control declares the planar isotropic coefficient forms

$$
\begin{aligned}
 m_K(u,v)&=(\rho u,v)_K,\\
 a_K(u,v)&=(\lambda\,\operatorname{div}u,\operatorname{div}v)_K
           +2(\mu\,\varepsilon(u),\varepsilon(v))_K.
\end{aligned}
$$

Common Basix nodal, quadrature, scatter and signed trace kernels assemble
these forms. The time algorithm is the shared free `newmark_step`, with
$\beta=1/4$ and $\gamma=1/2$, evaluated at every actual local substep.
The source callback supplies physical force density at the same local
endpoint times as the compatibility method.

For one macro time slab, $u_K^f,v_K^f$ are the free local histories and
$U_K,V_K$ are the responses to the constant signed trace force $Q_K\lambda$.
The declared local and global endpoint equations are

$$
\begin{aligned}
 u_K+U_K\lambda_K&=u_K^f,\\
 -\sum_KQ_K^\top u_K&=-b,\\
 v_K&=v_K^f-V_K\lambda_K.
\end{aligned}
$$

`LocalEquations` and `Equation` impose these actual responses through the
same generic global core. The multiplier is negative physical traction;
prescribed traction coefficients retain their physical sign conversion.
The reconstructed fields are also checked by replaying the original local
Newmark equations with their computed traction and identical integrated
endpoint loads. Field records provide evaluation and norms; a physical
Stepper does not select or advance the primary equations.

The inexpensive control below preserves the original free-body rigid
translation, mass, source and four steps. The additional control uses distinct
substep counts $(1,3)$ and nonhomogeneous displacement data. These executed
controls cover constant isotropic planar material. The archived 3D spatial,
temporal and native comparison campaigns below retain their original
provenance; arbitrary subcycling is not assigned a global energy theorem.

In [ ]:
import inspect
from IPython.display import Code
from examples.tutorial_elastodynamic_equations import (
    prepare, initialize, advance, spatial_forms,
    endpoint_equations, endpoint_global, projection_equations,
)
from pymhm.linalg.dynamics import newmark_step

for builder in (
    spatial_forms, newmark_step, endpoint_equations,
    endpoint_global, projection_equations,
):
    display(Code(inspect.getsource(builder), language="python"))

mesh = TriangleMesh.unit_square()
acceleration = np.array([1.0, -0.4])
density, dt = 2.4, 0.02
traction = {int(face): np.zeros(2) for face in mesh.boundary_faces}

def force(time: float, points: np.ndarray) -> np.ndarray:
    """The original constant physical body force rho times acceleration."""
    return density * acceleration

with threadpool_limits(1), prepare(
    mesh, time_step=dt, degree=2, density=density, traction=traction,
) as operator:
    solution = initialize(operator)
    trajectory = [solution]
    for _ in range(4):
        solution = advance(operator, solution, force)
        trajectory.append(solution)
        displacement_error = solution.l2_error(0.5 * solution.time**2 * acceleration)
        velocity_error = solution.l2_error(solution.time * acceleration, velocity=True)
        assert displacement_error < 1e-11
        assert velocity_error < 1e-10
{"time": solution.time, "displacement_l2_error": displacement_error,
 "velocity_l2_error": velocity_error,
 "free_face_constraint": solution.constraint_residual}

## Explicit coefficient and subcycling comparisons

The original Stepper is run separately for the same free-body data. A second
trajectory retains a distinct substep count in each macrocell, a temporal
source and constant nonzero displacement boundary. Both displacement and
velocity coefficient vectors, trace coordinates, physical norms, energy and
free-face constraints are compared at every common macro time.

In [ ]:
from pymhm._legacy.models.waves.elastodynamics import ElastodynamicStepper

with threadpool_limits(1), ElastodynamicStepper(
    mesh, time_step=dt, degree=2, density=density, traction=traction,
) as comparator:
    references = [comparator.initialize()]
    for _ in range(4):
        references.append(comparator.advance(force))
for own, reference in zip(trajectory, references, strict=True):
    for values, expected in zip(
        (*own.displacement, *own.velocity, own.trace),
        (*reference.displacement, *reference.velocity, reference.trace), strict=True,
    ):
        np.testing.assert_array_equal(values, expected)
    assert own.time == reference.time
    np.testing.assert_allclose(own.energy, reference.energy, atol=2e-15)
print("Original rigid-translation coefficient vectors are unchanged")

counts = (1, 3)
boundary_displacement = 0.03 * acceleration

def temporal_force(time: float, points: np.ndarray) -> np.ndarray:
    """Physical force evaluated at each actual local substep endpoint."""
    return density * (1 + 0.2 * time) * acceleration

options = dict(
    time_step=dt, degree=2, density=density, local_substeps=counts,
    dirichlet=boundary_displacement,
)
with threadpool_limits(1), prepare(mesh, **options) as operator, ElastodynamicStepper(
    mesh, **options
) as comparator:
    own = initialize(operator, boundary_displacement)
    reference = comparator.initialize(boundary_displacement)
    subcycling = [(own, reference)]
    for _ in range(4):
        own = advance(operator, own, temporal_force)
        reference = comparator.advance(temporal_force)
        subcycling.append((own, reference))
maximum_difference = 0.0
for own, reference in subcycling:
    for values, expected in zip(
        (*own.displacement, *own.velocity, own.trace),
        (*reference.displacement, *reference.velocity, reference.trace), strict=True,
    ):
        maximum_difference = max(maximum_difference, float(np.max(np.abs(values - expected))))
        np.testing.assert_allclose(values, expected, atol=2e-11, rtol=0)
    assert own.time == reference.time
    np.testing.assert_allclose(own.energy, reference.energy, atol=2e-13, rtol=0)
    np.testing.assert_allclose(own.constraint_residual, reference.constraint_residual, atol=2e-13)
    for velocity in (False, True):
        np.testing.assert_allclose(
            own.l2_error(boundary_displacement, velocity=velocity),
            reference.l2_error(boundary_displacement, velocity=velocity), atol=2e-13,
        )
{"local_substeps": counts, "maximum_coefficient_difference": maximum_difference}

## Persisted physical fields and norms

Every acquired field is checked against its recorded digest. Stress norms include the divergence inside each fine tetrahedron; this broken norm does not imply H(div)-conforming raw stress. The two quadrature rules used in the spatial campaign independently check the integrated errors.

In [ ]:
record = json.loads((data / "comparison.json").read_text())
assert [row["n"] for row in record["spatial"]] == [1, 2, 3, 4, 5, 6, 8]
for name, digest in record["field_sha256"].items():
    assert hashlib.sha256((data / name).read_bytes()).hexdigest() == digest
for row in record["spatial"]:
    assert row["quadrature_relative_change"] < 1e-10
    print(row["n"], row["macro_cells"], row["norms"])
print("Field digests and quadrature checks passed")

## Separate spatial and temporal errors

Spatial errors are measured against the analytical solution. Temporal differences use the same spatial space and a separately acquired $\Delta t=0.00009765625$ trajectory. Keeping these two comparisons separate prevents the spatial approximation floor from hiding the second-order time behavior. The slopes in the figure are references, not fitted error values.

In [ ]:
print("Time-reference refinement:", record["temporal_reference_refinement"])
temporal = record["temporal"]
assert len(temporal) == 8
steps = np.array([row["dt"] for row in temporal])
errors = np.array([row["difference_to_time_reference"]["displacement_l2"] for row in temporal])
print("Successive time orders:", np.log(errors[:-1] / errors[1:]) / np.log(steps[:-1] / steps[1:]))
display(Image(filename=str(root / "docs/figures/elastodynamics/convergence.png")))

## Displacement and velocity components

The section $z=0.37$ shows signed components, common analytical/numerical color limits and separate error scales. Lines are actual intersections of the macro tetrahedra with the section. The profile at $y=0.413,z=0.37$ retains independent one-sided polynomial values at fine and macro interfaces.

In [ ]:
for name in ("displacement-components", "velocity-components", "profiles"):
    display(Image(filename=str(root / "docs/figures/elastodynamics" / f"{name}.png")))

Independent DOLFINx/UFL full-system trajectories use the same P3/P1/Newmark spaces and Eq. (53) data. Nodal coordinates and permutations accompany the fields. These discrete comparisons complement the exact-solution convergence study.

In [ ]:
native = json.loads((root / "examples/results/elastodynamics/native/verification.json").read_text())
assert native["accepted"] and len(native["rows"]) == 5
[(row["n"], row["dt"], row["maximum_trajectory_difference"], row["physical_checks"]) for row in native["rows"]]

In [ ]:
for name in ("native-displacement", "native-velocity"):
    display(Image(filename=str(root / f"docs/figures/elastodynamics/{name}.png")))